# dscraft.vision quickstart

This notebook demonstrates `dscraft.vision`'s dense image pipeline (`SimpleImagePipeline`: decode via Pillow -> augment resize/flip -> dense tensor), a small `TinyCNN` classifier, and the `torch.export()` -> ONNX export path with correctness verification.

**Section 1** runs the pipeline/model/export flow on a small in-memory synthetic gradient image.

**Section 2** repeats the exact same flow on a second, differently-shaped synthetic image (a simple in-memory shape-and-gradient composite built with Pillow/NumPy) to show the pipeline handling more than one input.

This mirrors `packages/dscraft/examples/vision/export_cnn_example.py` in notebook form.

Requires only the `vision` extra (no `dev` extra, no scikit-learn -- both synthetic demo images are built in-memory from NumPy and Pillow, `vision`'s own required dependencies):
```bash
pip install -e "packages/dscraft[vision]"
```

In [1]:
import io
import logging
import tempfile
import warnings
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from dscraft.vision import (
    ModelConfig,
    PipelineConfig,
    SimpleImagePipeline,
    build_model,
    export_to_onnx,
    resolve_device,
    synthetic_classification_batch,
    verify_export,
)

# torch.onnx's exporter registration logger warns (at WARNING level, via
# stdlib logging, not warnings.warn) that a handful of torchvision-specific
# ops (nms/roi_align/roi_pool/deform_conv2d) aren't registered when
# torchvision isn't installed. TinyCNN never uses any of those ops, so this
# is a benign, irrelevant notice for this notebook -- silenced narrowly by
# raising just this logger's level, not global logging config.
logging.getLogger("torch.onnx").setLevel(logging.ERROR)

# torch.export's internal pytree handling triggers a known upstream
# FutureWarning (`isinstance(treespec, LeafSpec)` deprecated) from
# copyreg during tracing -- an upstream torch.export implementation detail
# unrelated to anything this notebook does. Suppressed narrowly by message,
# not by broadly ignoring all FutureWarnings.
warnings.filterwarnings("ignore", category=FutureWarning, message=".*LeafSpec.*")

IMAGE_SIZE = 32
NUM_CLASSES = 10

device = resolve_device()
print(f"Resolved device: {device}")

Resolved device: mps


## Section 1: pipeline + TinyCNN + torch.export -> ONNX on a synthetic image

In [2]:
def make_synthetic_image_bytes(size: int) -> bytes:
    """A small in-memory synthetic gradient-pattern PNG -- keeps this notebook hermetic."""
    array = np.zeros((size, size, 3), dtype=np.uint8)
    array[:, :, 0] = np.linspace(0, 255, size, dtype=np.uint8)[None, :]
    array[:, :, 1] = np.linspace(255, 0, size, dtype=np.uint8)[:, None]
    array[:, :, 2] = 100
    buf = io.BytesIO()
    Image.fromarray(array, mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


pipeline_config = PipelineConfig(image_size=IMAGE_SIZE, horizontal_flip_prob=0.5, seed=7, device=device)
pipeline = SimpleImagePipeline(pipeline_config)
raw_bytes = make_synthetic_image_bytes(size=40)

dense_tensor = pipeline.run(raw_bytes)
print(
    f"Pipeline produced a dense tensor: shape={tuple(dense_tensor.shape)}, "
    f"dtype={dense_tensor.dtype}, "
    f"range=[{dense_tensor.min():.3f}, {dense_tensor.max():.3f}]"
)

model_config = ModelConfig(in_channels=dense_tensor.shape[0], image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)
model = build_model(model_config, seed=0, device=device)
print(f"Built TinyCNN: {sum(p.numel() for p in model.parameters())} parameters")

trace_batch = dense_tensor.unsqueeze(0).repeat(2, 1, 1, 1)  # (2, C, H, W)

with tempfile.TemporaryDirectory() as tmp_dir:
    onnx_path = Path(tmp_dir) / "tiny_cnn.onnx"
    export_to_onnx(model, trace_batch, onnx_path)
    print(f"Exported ONNX model to: {onnx_path} ({onnx_path.stat().st_size} bytes)")

    verification_batch, _ = synthetic_classification_batch(model_config, batch_size=8, seed=123, device=device)
    result = verify_export(model, onnx_path, verification_batch, atol=1e-4, rtol=1e-3)

    print("\n=== Correctness check: PyTorch vs. ONNX Runtime ===")
    print(f"  matched:       {result.matched}")
    print(f"  max_abs_diff:  {result.max_abs_diff:.3e}")
    print(f"  max_rel_diff:  {result.max_rel_diff:.3e}")
    assert result.matched

Pipeline produced a dense tensor: shape=(3, 32, 32), dtype=torch.float32, range=[0.008, 0.992]
Built TinyCNN: 11642 parameters


[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX model to: /var/folders/bw/9966jv5s41zgvrvx6hpss6bh0000gn/T/tmp05atawi0/tiny_cnn.onnx (57793 bytes)



=== Correctness check: PyTorch vs. ONNX Runtime ===
  matched:       True
  max_abs_diff:  6.054e-08
  max_rel_diff:  1.873e-04


## Section 2: the same flow on a second synthetic image (shape-and-gradient composite)

In [3]:
from PIL import ImageDraw


def make_synthetic_shapes_image_bytes(size: int) -> bytes:
    """A second small in-memory synthetic image -- a vertical gradient
    background with simple geometric shapes drawn on top via
    `PIL.ImageDraw` -- keeps this notebook hermetic and runnable with only
    the `vision` extra installed (numpy + pillow), no scikit-learn/`dev`
    extra required. Mirrors the synthetic-gradient-image pattern already
    established in `packages/dscraft/tests/vision/test_pipeline.py`,
    combined with the `PIL.ImageDraw`-based synthetic shape generation used
    by dscraft's OCR notebook/tests."""
    array = np.zeros((size, size, 3), dtype=np.uint8)
    array[:, :, 0] = 40
    array[:, :, 1] = np.linspace(0, 255, size, dtype=np.uint8)[:, None]
    array[:, :, 2] = np.linspace(255, 0, size, dtype=np.uint8)[None, :]
    image = Image.fromarray(array, mode="RGB")

    draw = ImageDraw.Draw(image)
    margin = size // 8
    draw.ellipse((margin, margin, size - margin, size - margin), outline=(255, 255, 255), width=2)
    draw.rectangle((size // 3, size // 3, 2 * size // 3, 2 * size // 3), outline=(0, 0, 0), width=2)
    draw.line((0, 0, size, size), fill=(255, 255, 0), width=2)

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return buf.getvalue()


real_raw_bytes = make_synthetic_shapes_image_bytes(size=48)
print("Built a second synthetic image (gradient background + drawn shapes) in-memory via Pillow/NumPy.")

real_pipeline = SimpleImagePipeline(
    PipelineConfig(image_size=IMAGE_SIZE, horizontal_flip_prob=0.5, seed=7, device=device)
)
real_dense_tensor = real_pipeline.run(real_raw_bytes)
print(
    f"Pipeline produced a dense tensor: shape={tuple(real_dense_tensor.shape)}, "
    f"dtype={real_dense_tensor.dtype}, "
    f"range=[{real_dense_tensor.min():.3f}, {real_dense_tensor.max():.3f}]"
)

real_model_config = ModelConfig(in_channels=real_dense_tensor.shape[0], image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)
real_model = build_model(real_model_config, seed=0, device=device)

real_trace_batch = real_dense_tensor.unsqueeze(0).repeat(2, 1, 1, 1)

with tempfile.TemporaryDirectory() as tmp_dir:
    real_onnx_path = Path(tmp_dir) / "tiny_cnn_shapes.onnx"
    export_to_onnx(real_model, real_trace_batch, real_onnx_path)
    print(f"Exported ONNX model to: {real_onnx_path} ({real_onnx_path.stat().st_size} bytes)")

    real_verification_batch = real_dense_tensor.unsqueeze(0)
    real_result = verify_export(real_model, real_onnx_path, real_verification_batch, atol=1e-4, rtol=1e-3)

    print("\n=== Correctness check (second synthetic image): PyTorch vs. ONNX Runtime ===")
    print(f"  matched:       {real_result.matched}")
    print(f"  max_abs_diff:  {real_result.max_abs_diff:.3e}")
    print(f"  max_rel_diff:  {real_result.max_rel_diff:.3e}")
    assert real_result.matched

Built a second synthetic image (gradient background + drawn shapes) in-memory via Pillow/NumPy.
Pipeline produced a dense tensor: shape=(3, 32, 32), dtype=torch.float32, range=[0.000, 1.000]


[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX model to: /var/folders/bw/9966jv5s41zgvrvx6hpss6bh0000gn/T/tmp5uyigisc/tiny_cnn_shapes.onnx (57793 bytes)

=== Correctness check (second synthetic image): PyTorch vs. ONNX Runtime ===
  matched:       True
  max_abs_diff:  2.235e-08
  max_rel_diff:  2.718e-06


## Summary

This notebook ran `dscraft.vision`'s decode -> augment -> dense-tensor pipeline, built a `TinyCNN`, exported it via `torch.export()` -> ONNX, and verified correctness against the original PyTorch model -- first on a synthetic gradient image, then on a second synthetic image (a gradient background with drawn shapes), both built entirely in-memory via NumPy and Pillow (no scikit-learn, no `dev` extra required).

See `packages/dscraft/README.md`'s `## dscraft.vision` section for more detail.